In [1]:
import pandas as pd
import os
import re
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Load and preprocess data

In [2]:
data = pd.read_csv('poem_data_with_birth.csv', engine='python')

In [3]:
# Custom stopwords, nltk removes too much
stop_words = {
    'did', 'll', 'mustn', 'haven', 'both', 'all', 'a', 'only', 'same',
    's', 'aren', 'y', 'having', 'own', 'very', 'isn', 'had',
    'once', 'now', 'hasn', 'does', 'hadn',
    'but', 'shouldn', 'just', 'the', 'weren',
    'o', 'because', 'than', 've', 'been',
    'other', 'any', 'again', 'will',
    'wasn', 'itself', 're', 'do',
    'mightn', 'so', 'or',
    'each', 'too', 'needn', 'wouldn',
    'ma', 'an', 'being', 'can', 'won',
    'such', 'and', 'd', 'if', 'should',
    'are', 'am', 'some',
    'doing', 'has', 'its',
    'm', 'few', 'more', 'be',
    'myself', 'that', 'then',
    'most', 'while', 'was', 'is', 'ain'
}

In [4]:
LINESEP_RE = re.compile(r'\r+\n')
WHITESPACE_RE = re.compile(r'\s+')
TOKEN_RE = re.compile(r'<linesep>|\b\w+\b')

def preprocess(text):
    text = text.lower()
    text = WHITESPACE_RE.sub(' ', text)
    return TOKEN_RE.findall(text)

poems = data.iloc[:,2]
tokenized_poems = [preprocess(poem) for poem in poems]
poems_cleaned = [" ".join(poem) for poem in tokenized_poems]

# Generate vocabulary

In [5]:
poem_words = [
    word for poem in tokenized_poems for word in poem 
    if word not in stop_words and not (word == "<linesep>")
]

vocabulary = set(poem_words)

In [6]:
len(vocabulary)

105760

# Generate full TF-IDF vectors

In [7]:
vectorizer = TfidfVectorizer(
    stop_words = list(stop_words),
    max_df=0.75, 
    min_df=3, # there are some faultily meshed words in the dataset e.g. frogsare, try to elminate them by min_df
)
tf_idf_vecs = vectorizer.fit_transform(
    poems_cleaned
)

In [8]:
tf_idf_vecs.shape

(13745, 36453)

# Reduce dimensionality to 100 with SVD

In [9]:
svd = TruncatedSVD(n_components = 100, random_state = 42)
tf_idf_vecs_100d = svd.fit_transform(tf_idf_vecs)

In [10]:
tf_idf_vecs_100d.shape

(13745, 100)

In [11]:
explained_variance = svd.explained_variance_ratio_.sum()
print(f"Explained variance: {explained_variance:.2f}")

Explained variance: 0.14


# Test similarity

In [12]:
def get_most_similar_poem_idx(embeddings, base_idx):
    comparison_base = embeddings[base_idx]
    best_sim = -1
    best_idx = -1

    for i, emb in enumerate(embeddings):
        if i == base_idx:
            continue
        norm_base = np.linalg.norm(comparison_base)
        norm_emb = np.linalg.norm(emb)

        # skip zero vectors to avoid divide-by-zero
        if norm_base == 0 or norm_emb == 0:
            continue

        sim = np.dot(comparison_base, emb) / (norm_base * norm_emb)

        if sim > best_sim:
            best_sim = sim
            best_idx = i
    return best_idx, best_sim

In [16]:
base_idx = 1297
best_idx, best_sim = get_most_similar_poem_idx(tf_idf_vecs_100d, base_idx)

print("Most similar index:", best_idx)
print("Similarity:", best_sim)

Most similar index: 1298
Similarity: 0.8445607539941901


In [17]:
data.iloc[[base_idx, best_idx]]

,Title,Poet,Poem,Tags,Emotion,PoetBirth,Period
1297,Sonnet 133: Beshrew that heart that makes my h...,William Shakespeare,Beshrew that heart that makes my heart to groa...,"Love,Break-ups & Vexed Love,Unrequited Love,So...",Sadness,1564.0,1550-1780
1298,"Sonnet 139: O, call not me to justify the wrong",William Shakespeare,"O, call not me to justify the wrongThat thy un...","Love,Break-ups & Vexed Love,Unrequited Love",Anger,1564.0,1550-1780


# Save baseline embeddings

In [15]:
os.makedirs("poem_embeddings", exist_ok=True)
filename = "baseline.npy" 
vec_file = os.path.join("poem_embeddings", filename)
np.save(vec_file, tf_idf_vecs_100d)